# 05 — Fase 2 & 3: Algoritmi FCGR Ottimizzati, Analisi Visiva Multi-Scala e Benchmark Computazionale

## Obiettivi del Notebook
Questo notebook implementa e valida le **ottimizzazioni della rappresentazione FCGR** (`src/fcgr.py`) rispetto alla Baseline (`02_fcgr_generation.ipynb`):
1. **Verifica di Equivalenza Matematica Esatta (`max_abs_err = 0.0`)**: dimostriamo che l'algoritmo **Bitwise Vectorized** (`compute_fcgr_vectorized`) produce esattamente gli stessi conteggi del doppio ciclo Python della Baseline (`compute_fcgr`), ma con uno **speedup di 20x–35x**.
2. **Confronto Visivo al variare di $k \in \{4, 5, 6, 7, 8\}$ (`plot_multik_progression`)**: mostriamo sulla **stessa sequenza di eccDNA** come l'aumento di $k$ aumenti la precisione spaziale ($2^k \times 2^k$) ma faccia esplodere la sparsità fino a oltre il $99\%$ e il consumo di RAM.
3. **Rappresentazioni Ottimizzate Anti-Sparsità e Anti-Bias di Lunghezza (`plot_multiscale_fcgr_diagnostics`)**:
   - **Variante 1 (`Bitwise + uint8`)**: stessa FCGR $k=6$ calcolata bitwise e quantizzata a 8 bit (`4 KB` invece di `16 KB`, **-75% memoria**).
   - **Variante 2 (`RC + Log1p + uint8`)**: invarianza al filamento (*Reverse-Complement*) + compressione sub-lineare `log1p` per eliminare il bias di lunghezza tra sequenze corte e lunghe.
   - **Variante 3 (`Multi-Scala Frattale k=(4,5,6) + RC + Log1p + uint8`)**: stack a 3 canali `(3, 64, 64)` che allinea frattalmente il contesto denso $k=4$ (`16x16 -> 64x64`), il contesto intermedio $k=5$ (`32x32 -> 64x64`) e i dettagli fini $k=6$ (`64x64`), riducendo la sparsità spaziale da **~90% a <20%** e occupando solo **`12 KB`** in `uint8` (**-25% memoria** rispetto ai `16 KB` della baseline `float32`!).
4. **Benchmark Computazionale Completo (Fase 3)**: confronto sistematico di tempi (`ms/seq`), throughput (`seq/s`), memoria (`KB/MB/GB`) e sparsità su tutto il campione stratificato da 900 sequenze per $k \in [4, 10]$.

In [ ]:
# Su Google Colab esegui il mount del Drive:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Rilevamento automatico Google Colab vs Ambiente Locale
COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_fcgr_siamese')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
COMPARISON_DIR = ARTIFACTS_DIR / 'final_fcgr_comparison'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / 'eccdna_disease_detection_metadata.tsv'
FASTA_PATH = RAW_DIR / 'eccdna_disease_detection.body.fa'
MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'

print('PROJECT_ROOT :', PROJECT_ROOT)
print('MANIFEST_PATH:', MANIFEST_PATH, '-> exists:', MANIFEST_PATH.exists())
print('FASTA_PATH   :', FASTA_PATH, '-> exists:', FASTA_PATH.exists())

from src.fcgr import (
    load_fasta_sequences,
    reverse_complement,
    compute_fcgr,
    normalize_fcgr,
    compute_fcgr_vectorized,
    normalize_fcgr_log,
    quantize_fcgr_uint8,
    dequantize_fcgr_uint8,
    compute_multiscale_fcgr,
    compute_fcgr_diagnostics,
    plot_multik_progression,
    plot_multiscale_fcgr_diagnostics,
    plot_fcgr_comparison,
)

## 1. Test di Equivalenza Matematica Esatta e Speedup Preliminare (`compute_fcgr` vs `compute_fcgr_vectorized`)
Verifichiamo che la codifica bitwise tramite Lookup Table (`(cy << k) | cx`) riproduca **bit per bit** la matrice della baseline per ogni $k \in [4, 10]$.

In [ ]:
rng = np.random.default_rng(42)
test_seq = ''.join(rng.choice(list('ACGTACGTACGTN'), size=1500))

eq_rows = []
for k_val in range(4, 11):
    # Warm-up LUT
    _ = compute_fcgr_vectorized(test_seq, k_val)

    t0 = time.perf_counter()
    c_base, v_base, d_base = compute_fcgr(test_seq, k_val, return_distinct=True)
    t_base_ms = (time.perf_counter() - t0) * 1000.0

    t1 = time.perf_counter()
    c_vec, v_vec, d_vec = compute_fcgr_vectorized(test_seq, k_val, return_distinct=True)
    t_vec_ms = (time.perf_counter() - t1) * 1000.0

    max_diff = int(np.max(np.abs(c_base - c_vec)))
    assert max_diff == 0 and v_base == v_vec and d_base == d_vec, f'Discrepanza trovata a k={k_val}!'
    eq_rows.append({
        'k': k_val,
        'grid_shape': f'{1<<k_val}x{1<<k_val}',
        'valid_kmers': v_base,
        'distinct_kmers': d_base,
        'max_abs_diff': max_diff,
        'baseline_ms': round(t_base_ms, 3),
        'vectorized_ms': round(t_vec_ms, 3),
        'speedup_x': round(t_base_ms / max(t_vec_ms, 1e-6), 1),
    })

eq_df = pd.DataFrame(eq_rows)
print('=== VERIFICA EQUIVALENZA MATEMATICA ESATTA (ERRORE = 0) ===')
display(eq_df)

## 2. Caricamento Campione Reale eccDNA e Visualizzazione Progressione Multi-$k$ ($k = 4, 5, 6, 7, 8$)
Carichiamo il manifest binario e visualizziamo **la stessa sequenza reale di eccDNA** al variare di $k \in \{4, 5, 6, 7, 8\}$.
Questo grafico mostra chiaramente per la tesi perché aumentare $k$ oltre 6 doni maggiore risoluzione spaziale ($2^k \times 2^k$), ma al prezzo di una matrice quasi interamente vuota ($>97\% - 99\%$ di zeri).

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH, sep='\t', low_memory=False)
manifest['id'] = manifest['id'].astype(str)
manifest['class_id'] = manifest['class_id'].astype(int)

# Campioniamo 900 sequenze stratificate (2 classi x 3 fasce lunghezza x 150) come nel Notebook 02
manifest['len_bin'] = pd.qcut(manifest['length'], q=3, labels=['short', 'medium', 'long'])
pilot_parts = []
for (cid, lbin), grp in manifest.groupby(['class_id', 'len_bin'], observed=False):
    pilot_parts.append(grp.sample(n=min(len(grp), 150), random_state=42 + int(cid)))
pilot_df = pd.concat(pilot_parts, ignore_index=True)

print(f'Caricamento di {len(pilot_df)} sequenze campione dal FASTA...')
pilot_seqs = load_fasta_sequences(FASTA_PATH, pilot_df['id'].tolist())
print(f'Sequenze caricate: {len(pilot_seqs)}')

# Selezioniamo 1 Hotspot e 1 Stabile di lunghezza mediana (~450-550 bp) per l\'ispezione visiva
median_hotspot = pilot_df[(pilot_df['class_id'] == 1) & (pilot_df['len_bin'] == 'medium')].iloc[0]
median_stable = pilot_df[(pilot_df['class_id'] == 0) & (pilot_df['len_bin'] == 'medium')].iloc[0]

for sample_row, lbl in [(median_hotspot, 'Hotspot (1)'), (median_stable, 'Stable (0)')]:
    sid = str(sample_row['id'])
    plot_multik_progression(
        sequence=pilot_seqs[sid],
        ks=(4, 5, 6, 7, 8),
        sample_id=sid,
        label_name=lbl,
        save_path=COMPARISON_DIR / f'multik_progression_{sid}.png',
    )

## 3. Visualizzazione della Rappresentazione FCGR Multi-Scala Ottimizzata (`k=(4, 5, 6)` + RC + Log1p)
Illustriamo ora come funziona la **FCGR Multi-Scala Frattale (`compute_multiscale_fcgr`)**:
- Grazie all'autosimilarità del Chaos Game, ogni cella della matrice $k=4$ ($16 \times 16$) corrisponde esattamente a un quadrante $4 \times 4$ della matrice $k=6$ ($64 \times 64$), e ogni cella di $k=5$ ($32 \times 32$) corrisponde a un sotto-quadrante $2 \times 2$ di $k=6$.
- Sovrapponendo i 3 canali allineati `(k=4, k=5, k=6)` (con filamento Reverse-Complement e compressione `log1p`), la CNN riceve in ogni coordinata $(y, x)$ sia la statistica locale densa ($k=4, 5$) sia il dettaglio fine ($k=6$).
- Inoltre, quantizzando ciascun canale in `uint8` (`1 byte` per cella), l'intero stack a 3 canali occupa **`12 KB`** contro i **`16 KB`** della singola FCGR baseline in `float32` (**-25% di memoria** e sparsità spaziale ridotta da ~90% a <20%!).

In [ ]:
for sample_row, lbl in [(median_hotspot, 'Hotspot (1)'), (median_stable, 'Stable (0)')]:
    sid = str(sample_row['id'])
    seq = pilot_seqs[sid]
    _ = plot_multiscale_fcgr_diagnostics(
        sequence=seq,
        ks=(4, 5, 6),
        target_k=6,
        sample_id=sid,
        label_name=lbl,
        save_path=COMPARISON_DIR / f'multiscale_fcgr_{sid}.png',
    )

# Confronto diretto numerico/visivo Baseline vs Bitwise RC+Log (1ch uint8) vs Multi-Scala (3ch uint8)
sid_demo = str(median_hotspot['id'])
seq_demo = pilot_seqs[sid_demo]

t0 = time.perf_counter()
c_b, _, _ = compute_fcgr(seq_demo, 6)
f_b = normalize_fcgr(c_b)
t_b = (time.perf_counter() - t0) * 1000.0

t1 = time.perf_counter()
c_rc, _, _ = compute_fcgr_vectorized(seq_demo, 6, include_reverse_complement=True)
f_rc_u8 = quantize_fcgr_uint8(normalize_fcgr_log(c_rc))
t_rc = (time.perf_counter() - t1) * 1000.0

t2 = time.perf_counter()
ms_f32, _ = compute_multiscale_fcgr(seq_demo, ks=(4, 5, 6), target_k=6, include_reverse_complement=True, log_normalize=True)
ms_u8 = quantize_fcgr_uint8(ms_f32)
t_ms = (time.perf_counter() - t2) * 1000.0

_ = plot_fcgr_comparison(
    fcgr_baseline=f_b,
    fcgr_optimized=dequantize_fcgr_uint8(f_rc_u8),
    time_baseline_ms=t_b,
    time_optimized_ms=t_rc,
    name_baseline='Baseline k=6 (float32, 16 KB)',
    name_optimized='Opt 1: Bitwise RC+Log (uint8, 4 KB)',
    bytes_baseline=f_b.nbytes,
    bytes_optimized=f_rc_u8.nbytes,
    save_path=COMPARISON_DIR / 'comparison_baseline_vs_rclog.png',
)

_ = plot_fcgr_comparison(
    fcgr_baseline=f_b,
    fcgr_optimized=dequantize_fcgr_uint8(ms_u8),
    time_baseline_ms=t_b,
    time_optimized_ms=t_ms,
    name_baseline='Baseline k=6 (1ch float32, 16 KB)',
    name_optimized='Opt 2: Multi-Scala k=4,5,6 (3ch uint8, 12 KB)',
    bytes_baseline=f_b.nbytes,
    bytes_optimized=ms_u8.nbytes,
    save_path=COMPARISON_DIR / 'comparison_baseline_vs_multiscale.png',
)

## 4. Benchmark Computazionale Sistematico (Fase 3 — 900 Sequenze Stratificate)
Eseguiamo il benchmark completo su tutte le 900 sequenze del campione stratificato:
1. **Scalabilità per $k \in [4, 10]$**: confronto tra `Baseline (Python loop, float32)` e `Vectorized Bitwise (NumPy, uint8)`.
2. **Confronto delle 4 Varianti a $k=6$ (risoluzione $64 \times 64$ per `SiameseV3`)**:
   - `Baseline_k6_float32` (Ciclo Python, 1 canale `float32`, `16 KB/seq`, `915.5 MB` per 60k seq)
   - `Bitwise_k6_uint8` (Bitwise NumPy, 1 canale `uint8`, `4 KB/seq`, `228.9 MB` per 60k seq)
   - `Bitwise_RC_Log_k6_uint8` (Bitwise + Reverse-Complement + Log1p, 1 canale `uint8`, `4 KB/seq`, `228.9 MB`)
   - `MultiScale_k456_RC_Log_uint8` (Bitwise Multi-Scala $k=4,5,6$ + RC + Log1p, 3 canali `uint8`, `12 KB/seq`, `686.6 MB`)

In [ ]:
N_CACHE_TOTAL = 60_000  # 50k train + 5k val + 5k test
k_bench_rows = []

for k_val in range(4, 11):
    side = 1 << k_val
    cells = side * side
    base_times, vec_times, base_sparsities = [], [], []

    for sid, seq in pilot_seqs.items():
        t0 = time.perf_counter()
        c_b, _, d_b = compute_fcgr(seq, k_val, return_distinct=True)
        base_times.append((time.perf_counter() - t0) * 1000.0)
        base_sparsities.append((1.0 - d_b / float(cells)) * 100.0)

        t1 = time.perf_counter()
        c_v, _, _ = compute_fcgr_vectorized(seq, k_val, return_distinct=True)
        vec_times.append((time.perf_counter() - t1) * 1000.0)

    mean_base_ms = float(np.mean(base_times))
    mean_vec_ms = float(np.mean(vec_times))
    k_bench_rows.append({
        'k': k_val,
        'fcgr_shape': f'{side}x{side}',
        'median_sparsity_pct': round(float(np.median(base_sparsities)), 2),
        'baseline_mean_ms': round(mean_base_ms, 4),
        'vectorized_mean_ms': round(mean_vec_ms, 4),
        'speedup_x': round(mean_base_ms / max(mean_vec_ms, 1e-6), 2),
        'mem_float32_KB': round((cells * 4) / 1024.0, 2),
        'mem_uint8_KB': round((cells * 1) / 1024.0, 2),
        'cache_60k_float32_GB': round((N_CACHE_TOTAL * cells * 4) / (1024 ** 3), 3),
        'cache_60k_uint8_GB': round((N_CACHE_TOTAL * cells * 1) / (1024 ** 3), 3),
    })

k_bench_df = pd.DataFrame(k_bench_rows)
k_bench_df.to_csv(RESULTS_DIR / 'optimized_fcgr_k_benchmark.csv', index=False)
print('=== BENCHMARK SCALABILITA\' PER k (BASELINE vs BITWISE VECTORIZED + UINT8) ===')
display(k_bench_df)

In [ ]:
# Benchmark specifico delle 4 Varianti candidate a k=6 (64x64)
variant_records = []

for sid, seq in pilot_seqs.items():
    # 1. Baseline k=6 float32
    t0 = time.perf_counter()
    cb, _, db = compute_fcgr(seq, 6, return_distinct=True)
    fb = normalize_fcgr(cb)
    tb = (time.perf_counter() - t0) * 1000.0
    sp_b = (1.0 - db / 4096.0) * 100.0
    variant_records.append({'variant': '1_Baseline_k6_float32', 'ms': tb, 'spatial_sparsity_pct': sp_b, 'kb_per_seq': 16.0, 'channels': 1})

    # 2. Bitwise k=6 uint8
    t0 = time.perf_counter()
    cv, _, dv = compute_fcgr_vectorized(seq, 6, return_distinct=True, include_reverse_complement=False)
    fv_u8 = quantize_fcgr_uint8(normalize_fcgr(cv))
    tv = (time.perf_counter() - t0) * 1000.0
    sp_v = (1.0 - dv / 4096.0) * 100.0
    variant_records.append({'variant': '2_Bitwise_k6_uint8', 'ms': tv, 'spatial_sparsity_pct': sp_v, 'kb_per_seq': 4.0, 'channels': 1})

    # 3. Bitwise RC + Log1p k=6 uint8
    t0 = time.perf_counter()
    crc, _, drc = compute_fcgr_vectorized(seq, 6, return_distinct=True, include_reverse_complement=True)
    frc_u8 = quantize_fcgr_uint8(normalize_fcgr_log(crc))
    trc = (time.perf_counter() - t0) * 1000.0
    sp_rc = (1.0 - drc / 4096.0) * 100.0
    variant_records.append({'variant': '3_Bitwise_RC_Log_k6_uint8', 'ms': trc, 'spatial_sparsity_pct': sp_rc, 'kb_per_seq': 4.0, 'channels': 1})

    # 4. Multi-Scale k=(4,5,6) RC + Log1p uint8
    t0 = time.perf_counter()
    ms, _ = compute_multiscale_fcgr(seq, ks=(4, 5, 6), target_k=6, include_reverse_complement=True, log_normalize=True)
    ms_u8 = quantize_fcgr_uint8(ms)
    tms = (time.perf_counter() - t0) * 1000.0
    active_px = int(np.count_nonzero(np.any(ms_u8 > 0, axis=0)))
    sp_ms = (1.0 - active_px / 4096.0) * 100.0
    variant_records.append({'variant': '4_MultiScale_k456_RC_Log_uint8', 'ms': tms, 'spatial_sparsity_pct': sp_ms, 'kb_per_seq': 12.0, 'channels': 3})

var_df = pd.DataFrame(variant_records)
var_summary = var_df.groupby('variant', as_index=False).agg(
    channels=('channels', 'first'),
    mean_time_ms=('ms', 'mean'),
    median_spatial_sparsity_pct=('spatial_sparsity_pct', 'median'),
    kb_per_seq=('kb_per_seq', 'first'),
)
base_ms_ref = float(var_summary.loc[var_summary['variant'] == '1_Baseline_k6_float32', 'mean_time_ms'].iloc[0])
base_kb_ref = 16.0
var_summary['speedup_vs_baseline_x'] = (base_ms_ref / var_summary['mean_time_ms']).round(2)
var_summary['memory_saving_pct'] = ((1.0 - var_summary['kb_per_seq'] / base_kb_ref) * 100.0).round(1)
var_summary['cache_60k_MB'] = ((var_summary['kb_per_seq'] * N_CACHE_TOTAL) / 1024.0).round(1)
var_summary['est_60k_gen_seconds'] = ((var_summary['mean_time_ms'] * N_CACHE_TOTAL) / 1000.0).round(1)

var_summary.to_csv(RESULTS_DIR / 'optimized_variants_k6_summary.csv', index=False)
print('=== RIEPILOGO VARIANTI CANDIDATE A k=6 (64x64) PER LA RETE SIAMESE ===')
display(var_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

axes[0].plot(k_bench_df['k'], k_bench_df['baseline_mean_ms'], marker='o', linewidth=2, label='Baseline (Ciclo Python)', color='#C53030')
axes[0].plot(k_bench_df['k'], k_bench_df['vectorized_mean_ms'], marker='s', linewidth=2, label='Bitwise Vectorized (NumPy)', color='#2B6CB0')
axes[0].set_xlabel('Ordine k')
axes[0].set_ylabel('Tempo Medio per Sequenza (ms)')
axes[0].set_title('Tempo di Calcolo FCGR: Baseline vs Bitwise')
axes[0].legend()
axes[0].grid(alpha=0.3)

labels_short = ['Baseline\n(1ch f32)', 'Bitwise\n(1ch u8)', 'RC+Log\n(1ch u8)', 'Multi-Scale\n(3ch u8)']
colors = ['#C53030', '#3182CE', '#2B6CB0', '#2F855A']
axes[1].bar(labels_short, var_summary['cache_60k_MB'], color=colors)
axes[1].set_ylabel('Dimensione Cache 60.000 Sequenze (MB)')
axes[1].set_title('Occupazione Memoria (RAM / Disco)')
for i, v in enumerate(var_summary['cache_60k_MB']):
    axes[1].text(i, v + 15, f'{v:.0f} MB', ha='center', fontweight='bold', fontsize=9.5)
axes[1].set_ylim(0, max(var_summary['cache_60k_MB']) * 1.15)
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(labels_short, var_summary['median_spatial_sparsity_pct'], color=colors)
axes[2].set_ylabel('Sparsità Spaziale Mediana (% pixel ciechi)')
axes[2].set_title('Abbattimento della Sparsità Spaziale (64x64)')
for i, v in enumerate(var_summary['median_spatial_sparsity_pct']):
    axes[2].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=9.5)
axes[2].set_ylim(0, 105)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'phase3_computational_benchmark.png', dpi=300, bbox_inches='tight')
plt.show()